# Simple RAG with Filtering + Query Transformations

## 적용 기법
- **Metadata Filtering** : category_large / category_medium 기반 2단계 장르 필터
- **Query Transformations** : Sub-query Decomposition → 서브쿼리별 검색결합


## 0. 설치 및 초기화

In [ ]:
!git clone https://github.com/jjeong3150/AIFFEL_final_pjt_book-recommendation-agent

Cloning into 'AIFFEL_final_pjt_book-recommendation-agent'...
remote: Enumerating objects: 434, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (164/164), done.
remote: Total 434 (delta 107), reused 0 (delta 0), pack-reused 270 (from 1)
Receiving objects: 100% (434/434), 26.03 MiB | 14.63 MiB/s, done.
Resolving deltas: 100% (233/233), done.


In [2]:
!pip install -q qdrant-client
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-naver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 550.1/550.1 kB 18.0 MB/s eta 0:00:00


In [ ]:
%run /content/AIFFEL_final_project_peekabook/research/src/state/state_v2.ipynb
%run /content/AIFFEL_final_project_peekabook/research/src/db/qdrant.py
%run /content/AIFFEL_final_project_peekabook/research/src/embedding/embedder.py
%run /content/AIFFEL_final_project_peekabook/research/src/reranking/reranker.py

# 카테고리 트리 로드
import pandas as pd

_df = pd.read_csv("/content/AIFFEL_final_project_peekabook/research/src/rag/query_transformations/aladin_category.csv")

CATEGORY_TREE = (
    _df.groupby("category_large")["category_medium"]
    .apply(lambda x: sorted(x.unique().tolist()))
    .to_dict()
)

CATEGORY_LARGE_LIST = sorted(CATEGORY_TREE.keys())


In [4]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"]    = userdata.get('OPENAI_API_KEY')
os.environ["CLOVASTUDIO_API_KEY"] = userdata.get('CLOVASTUDIO_API_KEY')
os.environ["QDRANT_API_KEY"]    = userdata.get('QDRANT_API_KEY')
os.environ["QDRANT_URL"]        = userdata.get('QDRANT_URL')

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_naver import ChatClovaX

embedder = LocalEmbedder("BAAI/bge-m3")
db       = QdrantDB(vector_size=1024)
llm      = ChatOpenAI(model="gpt-4o-mini")
reranker = LocalReranker("BAAI/bge-reranker-v2-m3")
# llm    = ChatClovaX(model="HCX-005", temperature=0)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

## 1. 장르 추출 노드 — 2단계 계층 추출 (대분류 → 중분류)


In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from qdrant_client.models import Filter, FieldCondition, MatchAny
import json

top_genre_prompt = ChatPromptTemplate.from_template("""
사용자 프로파일을 보고 아래 대분류 목록에서 적합한 것을 최대 2개 선택하세요.
목록에 없는 값은 절대 반환하지 마세요.

대분류 목록: {large_list}
사용자 프로파일: {summary}

JSON으로만 반환: {{"categories": ["소설/시/희곡"]}}
""")

medium_genre_prompt = ChatPromptTemplate.from_template("""
사용자 프로파일을 보고 아래 중분류 목록에서 적합한 것을 최대 3개 선택하세요.
목록에 없는 값은 절대 반환하지 마세요.

중분류 목록: {medium_list}
사용자 프로파일: {summary}

JSON으로만 반환: {{"categories": ["한국소설"]}}
""")

def extract_genre_node(state: GraphState) -> dict:
    summary = state.get("summary", "")

    # ── 1단계: 대분류 (30개) ──────────────────────────────────
    top_resp = (top_genre_prompt | llm).invoke({
        "large_list": CATEGORY_LARGE_LIST,
        "summary": summary,
    })
    try:
        top_cats = json.loads(top_resp.content)["categories"]
    except (json.JSONDecodeError, KeyError):
        top_cats = []

    if not top_cats:
        print("[Genre] 대분류 추출 실패 → 필터 없음")
        return {"genre_filter": [], "genre_level": "none"}

    # ── 2단계: 중분류 (해당 대분류 하위만) ───────────────────
    medium_candidates = []
    for cat in top_cats:
        medium_candidates.extend(CATEGORY_TREE.get(cat, []))

    if not medium_candidates:
        print(f"[Genre] 대분류 fallback: {top_cats}")
        return {"genre_filter": top_cats, "genre_level": "large"}

    medium_resp = (medium_genre_prompt | llm).invoke({
        "medium_list": medium_candidates,
        "summary": summary,
    })
    try:
        medium_cats = json.loads(medium_resp.content)["categories"]
    except (json.JSONDecodeError, KeyError):
        medium_cats = []

    if not medium_cats:
        print(f"[Genre] 중분류 추출 실패 → 대분류 fallback: {top_cats}")
        return {"genre_filter": top_cats, "genre_level": "large"}

    print(f"[Genre] 대분류: {top_cats} → 중분류: {medium_cats}")
    return {"genre_filter": medium_cats, "genre_level": "medium"}


## 2. Step-back Prompting
**입력**: 원본 사용자 프로파일  
**출력**: 추상화된 독서 목적  
**목적**: 구체적인 프로파일에서 한 단계 물러나 독자 타겟·독서 목적을 추상화 → 의도 파악 문제 해결

In [ ]:
step_back_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 AI 어시스턴트입니다.
아래 사용자 프로파일에서 한 단계 물러나,
더 넓은 범위의 도서를 검색할 수 있는 일반적인 쿼리를 생성하세요.
특정 장르나 조건에 국한되지 않고 독서 경험과 목적 중심으로 작성하세요.
2문장 이내로 작성하세요.

사용자 프로파일: {summary}

출력:
""")

def step_back_query(summary: str, llm) -> str:
    """
    사용자 프로파일 → 추상화된 독서 목적

    Args:
        summary : 원본 사용자 프로파일
        llm     : LLM 인스턴스

    Returns:
        str: 추상화된 독서 목적
    """
    chain = step_back_prompt | llm
    return chain.invoke({"summary": summary}).content.strip()

## 3. Query Rewriting
**입력**: step_back 결과 (추상화된 독서 목적)  
**출력**: 검색에 최적화된 구체적인 쿼리  
**목적**: 추상화된 의도를 도서 검색에 더 적합한 표현으로 구체화

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 AI 어시스턴트입니다.
아래 독서 목적을 도서 검색에 더 적합하고 구체적인 검색어로 재작성하세요.
장르, 독자 수준, 분위기 등 검색 정확도를 높일 수 있는 표현을 포함하세요.

독서 목적: {step_back}

재작성된 검색 쿼리 (두 문장 이내로):
""")

def rewrite_query(step_back: str, llm) -> str:
    """
    추상화된 독서 목적 → 검색에 최적화된 구체적 쿼리

    Args:
        step_back : step_back_query()의 결과
        llm       : LLM 인스턴스

    Returns:
        str: 재작성된 검색 쿼리
    """
    chain = rewrite_prompt | llm
    return chain.invoke({"step_back": step_back}).content.strip()

## 4. Sub-query Decomposition
**입력**: rewritten 결과 (구체화된 쿼리)  
**출력**: 조건 단위로 분해된 서브쿼리 리스트  
**목적**: 다중 조건을 분해하여 각각 검색 → RRF 결합으로 조건 만족률 향상

In [ ]:
decompose_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 검색 쿼리 전문가입니다.
사용자의 독서 취향과 상황을 깊이 이해하여, 벡터 임베딩 검색에 최적화된 서브쿼리를 생성합니다.

아래 검색 쿼리를 2~4개의 서브쿼리로 분해하세요.
각 서브쿼리는 독립적인 문장으로 작성하되, 전체적으로 자연스럽게 맥락이 이어지도록 하세요.
원래 쿼리의 조건을 충실히 반영하면서, 사용자가 미처 생각하지 못했을 관련 관점을 한 개 포함하세요.


[주의]
- "이 중에서", "그 중에서" 같은 참조 표현은 사용하지 마세요.
- 리뷰, 평점, 사용자 의견 등 도서 메타데이터 외의 정보를 요청하지 마세요.
- "추천 도서", "추천해주세요", "포함해주세요" 같은 표현으로 끝내지 마세요.

검색 쿼리: {rewritten}

출력 형식 (번호와 텍스트만, 다른 텍스트 없이):
1. [서브쿼리 1]
2. [서브쿼리 2]
3. [서브쿼리 3]
""")


def decompose_query(rewritten: str, llm) -> list:
    """
    구체화된 쿼리 → 서브쿼리 리스트

    Args:
        rewritten : rewrite_query()의 결과
        llm       : LLM 인스턴스

    Returns:
        List[str]: 서브쿼리 리스트
    """
    chain = decompose_prompt | llm
    response = chain.invoke({"rewritten": rewritten}).content
    sub_queries = [
        q.strip().lstrip("1234567890. ")
        for q in response.split("\n")
        if q.strip() and q.strip()[0].isdigit()
    ]
    return sub_queries

## 5. Chained Pipeline
세 단계를 순서대로 이어서 실행하는 메인 **함수**

In [ ]:
def get_chained_queries(summary: str, llm,
                        use_step_back: bool = True,
                        use_rewrite: bool = True,
                        use_decompose: bool = True) -> dict:
    """
    세 가지 변환을 순서대로 이어서 실행

    흐름: summary → step_back → rewritten → sub_queries

    Args:
        summary       : 원본 사용자 프로파일
        llm           : LLM 인스턴스
        use_step_back : Step-back Prompting 사용 여부 (기본 True)
        use_rewrite   : Query Rewriting 사용 여부 (기본 True)
        use_decompose : Sub-query Decomposition 사용 여부 (기본 True)

    Returns:
        {
            'step_back'  : str,       # Step-back 결과
            'rewritten'  : str,       # Rewriting 결과
            'sub_queries': List[str], # Decomposition 결과
            'all'        : List[str], # RRF에 넘길 전체 쿼리 목록
        }
    """
    # 1단계: Step-back → 의도 추상화
    step_back = step_back_query(summary, llm) if use_step_back else summary
    print(f"  [Step-back]  : {step_back}")

    # 2단계: Rewriting → 추상화된 의도를 구체적 검색어로
    rewritten = rewrite_query(summary, llm) if use_rewrite else summary
    print(f"  [Rewritten]  : {rewritten}")

    # 3단계: Decomposition → 구체적 쿼리를 조건 단위로 분해
    sub_queries = decompose_query(rewritten, llm) if use_decompose else []
    print(f"  [Sub-queries]: {sub_queries}")

    # 전체 쿼리 목록 (RRF 입력용)
    all_queries = [step_back, rewritten] + sub_queries

    return {
        "step_back":   step_back,
        "rewritten":   rewritten,
        "sub_queries": sub_queries,
        "all":         all_queries,
    }

## 6. 테스트 실행

In [ ]:
# # TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
# TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."

# print("[Query Transformations 실행]")
# queries = get_chained_queries(TEST_SUMMARY, llm)

# print(f"\n[전체 쿼리 목록] 총 {len(queries['all'])}개")
# for i, q in enumerate(queries['all'], 1):
#     print(f"  {i}. {q}")

## 7. Ablation Study용 — 단계별 조합 테스트

In [ ]:
# # Sub-query만 사용
# print("[Sub-query only]")
# q1 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=False,
#                           use_rewrite=False,
#                           use_decompose=True)

# # Step-back + Sub-query
# print("\n[Step-back + Sub-query]")
# q2 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=True,
#                           use_rewrite=False,
#                           use_decompose=True)

# # 전체 (Step-back + Rewriting + Sub-query)
# print("\n[전체 파이프라인]")
# q3 = get_chained_queries(TEST_SUMMARY, llm,
#                           use_step_back=True,
#                           use_rewrite=True,
#                           use_decompose=True)

## 8. RRF 함수 정의

In [ ]:
def reciprocal_rank_fusion(results_list: list, k: int = 60) -> list:
    """여러 검색 결과를 RRF로 결합. Returns: payload 딕셔너리 리스트 (점수 내림차순)"""
    scores, payloads = {}, {}
    for results in results_list:
        for rank, r in enumerate(results):
            isbn = r.payload.get("isbn", "")
            if isbn:
                scores[isbn]   = scores.get(isbn, 0) + 1 / (k + rank + 1)
                payloads[isbn] = r.payload
    return [payloads[isbn] for isbn, _ in sorted(scores.items(), key=lambda x: x[1], reverse=True)]

## 9. Query Transform RAG 노드

In [ ]:
def query_transform_rag_node(state: GraphState) -> dict:
    summary    = state.get("summary", "")
    reflection = state.get("reflection", "")
    categories = state.get("genre_filter", [])
    genre_level = state.get("genre_level", "none")

    # summary + reflection 결합
    user_profile_query = " ".join(filter(None, [summary, reflection]))

    # ── 1. Query Transformations ────────────────────────────
    print("\n[Query Transformations]")
    queries     = get_chained_queries(user_profile_query, llm)
    all_queries = queries["all"]

    # ── 2. 장르 필터 구성 ────────────────────────────────────
    field_map = {"large": "category_large", "medium": "category_medium"}
    query_filter = None
    if categories and genre_level in field_map:
        query_filter = Filter(
            must=[FieldCondition(key=field_map[genre_level], match=MatchAny(any=categories))]
        )

    # ── 3. 쿼리별 검색 ──────────────────────────────────────
    all_results = []
    for query in all_queries:
        query_vector = embedder.embed(query)
        if query_filter:
            results = db.search_with_filter(
                "books_v1", query_vector,
                query_filter=query_filter, limit=5, threshold=0.5,
            )
        else:
            results = db.search("books_v1", query_vector, limit=5, threshold=0.5)
        all_results.append(results)

        # print(query)
        # for r in results:
        #     print(f"score: {r.score}")
        #     print(f"title: {r.payload.get('title')}")
        #     print(f"book_intro: {r.payload.get('book_intro')}")
        # print("---------------")

    # ── 4. RRF 결합
    merged_payloads = reciprocal_rank_fusion(all_results)

    # ── 5. Reranker Model 적용
    reranked_payloads = reranker.rerank(query=user_profile_query, books=merged_payloads)

    # ── 6. 결과 정리
    retrieved_books = [
        {
            "isbn":            p.get("isbn"),
            "title":           p.get("title"),
            "author":          p.get("author"),
            "book_intro":      p.get("book_intro"),
            "category_large":  p.get("category_large"),
            "category_medium": p.get("category_medium"),
        }
        for p in reranked_payloads[:5]
    ]

    # print(f"\n최종 검색 결과: {len(retrieved_books)}권")
    return {"retrieved_books": retrieved_books}


## 10. Explain 노드
각 검색 도서에 대해 `summary + reflection + book_intro` 기반으로 사전 분석을 생성.

`rag_llm_node`가 이 분석을 근거로 추천 이유를 작성하도록 함.

In [ ]:
explain_prompt = ChatPromptTemplate.from_template("""
당신은 도서 추천 시스템의 AI 어시스턴트입니다.
아래 사용자 프로파일과 책 소개를 읽고,
이 책이 이 사용자에게 왜 적합한지 또는 적합하지 않은지 2문장으로 분석하세요.

[주의]
- 책 소개에 없는 내용은 절대 지어내지 마세요.
- 사용자 프로파일의 독서 목적, 선호 장르, 독서 스타일과 연결해서 작성하세요.

[사용자 프로파일]
{summary}

[책 소개]
{book_intro}

분석:
""")

explain_chain = explain_prompt | llm


def explain_node(state: GraphState) -> dict:
    summary = state.get("summary", "")
    reflection = state.get("reflection", "")
    books   = state["retrieved_books"]

    user_profile_query = " ".join(filter(None, [summary, reflection]))

    for book in books:
        analysis = explain_chain.invoke({
            "summary":    user_profile_query,
            "book_intro": book.get("book_intro", ""),
        }).content
        book["analysis"] = analysis
        print(f"  [분석] {book.get('title')} → {analysis[:60]}...")

    return {"retrieved_books": books}


## 11. LLM 노드



In [ ]:
def rag_llm_node(state: GraphState) -> dict:
    summary = state.get("summary", "")
    reflection = state.get("reflection", "")
    books   = state["retrieved_books"]

    user_profile_query = " ".join(filter(None, [summary, reflection]))

    context = "\n\n".join([
        f"ISBN: {b['isbn']}\n"
        f"제목: {b['title']}\n"
        f"저자: {b['author']}\n"
        f"장르: {b.get('category_medium') or b.get('category_large', '')}\n"
        f"소개: {b['book_intro'][:300]}\n"
        f"사전분석: {b.get('analysis', '')}"
        for b in books
    ])

    rag_prompt = ChatPromptTemplate.from_template("""
당신은 도서관 큐레이터 AI입니다.

[규칙]
- 반드시 [검색된 도서 목록]에 있는 책만 추천하세요.
- 반드시 JSON 형식으로만 답하세요. 다른 텍스트는 절대 포함하지 마세요.
- 사용자 프로파일을 참고해서 가장 적합한 도서 3권을 추천하세요.
- 장르는 반드시 [검색된 도서 목록]의 장르 값을 그대로 사용하세요.

[추천 이유 작성 규칙]
- 반드시 [사전분석]과 [소개]에 나온 구체적인 내용을 근거로 작성하세요.
- 사용자 프로파일의 어떤 부분(독서 목적, 선호 장르, 독서 스타일)과 연결되는지 명시하세요.
- [소개]와 [사전분석]에 없는 내용을 임의로 지어내지 마세요.
- 2~3문장으로 작성하세요.

[사용자 프로파일]
{summary}

[검색된 도서 목록]
{context}

[출력 형식]
[
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "category_medium": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "category_medium": "장르", "reason": "추천 이유 2~3문장"}},
    {{"title": "책 제목", "author": "저자", "isbn": "ISBN번호", "book_intro": "책 소개", "category_medium": "장르", "reason": "추천 이유 2~3문장"}}
]
""")

    chain    = rag_prompt | llm
    response = chain.invoke({"context": context, "summary": user_profile_query})

    try:
        recommendations = json.loads(response.content)
    except json.JSONDecodeError:
        recommendations = response.content

    return {
        "messages":        [AIMessage(content=response.content)],
        "recommendations": recommendations
    }


## 11. 그래프 구성 및 실행

In [ ]:
graph_test = StateGraph(GraphState)
graph_test.add_node("genre", extract_genre_node)
graph_test.add_node("rag",   query_transform_rag_node)
graph_test.add_node("explain", explain_node)
graph_test.add_node("llm",   rag_llm_node)

graph_test.add_edge(START,     "genre")
graph_test.add_edge("genre",   "rag")
graph_test.add_edge("rag",     "explain")
graph_test.add_edge("explain", "llm")
graph_test.add_edge("llm",     END)
app_test = graph_test.compile()

TEST_SUMMARY = "사용자는 한국 현대소설을 선호하며, 감성을 풍부하게 하고 다양한 시각을 배우고자 하는 목표를 가지고 있습니다. 현재 감성적이고 몽환적인 기분 속에서 잔잔한 음악과 함께 조용한 공간에서 독서를 즐기고 있으며, 스토리와 감정에 집중하는 스타일을 선호합니다. 중급 난이도의 작품을 통해 감정 정리와 아름다움을 느끼고자 하는 상황입니다."
# TEST_SUMMARY = "사용자는 주말에 가볍게 읽으면서 스트레스를 해소하고 새로운 아이디어를 얻고자 하며, SF와 테크 스릴러 장르의 흥미로운 이야기를 선호합니다. 독서 스타일은 빠르게 전체적인 흐름을 파악하고, 스토리의 흡입력에 집중하는 중급자용입니다. 깊이 있는 분석보다는 이야기 전개와 흥미로운 설정에 중점을 두고 있으며, 가벼운 책을 통해 재미를 추구하고 있습니다."
# TEST_SUMMARY = "사용자는 수업에서 활용할 수 있는 흥미로운 역사 이야기나 사례를 찾고 있으며, 역사 비문학 장르를 선호합니다. 천천히 깊이 있게 읽는 스타일로, 중요한 내용이나 흥미로운 부분을 메모하여 학생들에게 설명하는 데 활용하려고 합니다. 중급 난이도의 책을 선호하며, 학생들에게 긍정적인 영향을 줄 수 있는 내용을 중시하고 있습니다."

result_test = app_test.invoke({
    "messages":              [HumanMessage(content=TEST_SUMMARY)],
    "session_id":            "test_session",
    "phase":                 Phase.SLOT_FILLING,
    "turn_count":            0,
    "user_profile":          UserProfile(),
    "current_slot":          None,
    "similar_profiles":      [],
    "matched_profile_id":    None,
    "book_experiences":      [],
    "asked_book_experience": False,
    "summary":               TEST_SUMMARY,
    "reflection":            "",
    "links":                 [],
    "ai_response":           "",
    "retrieved_books":       [],
    "recommendations":       [],
    "genre_filter":          [],
    "genre_level":           "none",
    "availability_results":  None,
})

print(result_test["messages"][-1].content)


추출된 장르: ['소설']

[Query Transformations]
  [Step-back]  : 현재의 감성과 정서를 깊이 탐구할 수 있는 다양한 현대 문학 작품들, 특히 서정적이고 감동적인 이야기를 중심으로 추천해 주세요. 삶과 인간 관계의 복잡함을 조사하는 글들로, 독자가 감정에 동화될 수 있는 경험을 할 수 있도록 해주시면 좋겠습니다.
  [Rewritten]  : 한국 현대소설 중 감성이 풍부하고 몽환적인 분위기를 가진 중급 난이도 작품으로, 스토리와 감정에 집중할 수 있는 감정 정리와 아름다움을 느낄 수 있는 도서를 추천해주세요. 잔잔한 음악과 함께 독서하기 적합한 작품을 찾고 있습니다.
  [Sub-queries]: ['한국 현대소설 중에서 감성이 풍부하고 몽환적인 분위기를 가진 중급 난이도의 작품을 찾고 있습니다.', '스토리와 감정에 집중할 수 있으며, 감정 정리와 아름다움을 느낄 수 있는 도서를 알아보세요.', '잔잔한 음악과 함께 독서하기 적합한 작품으로, 독특한 서사 구조를 가진 책을 포함해서 찾아주세요.']

[Reranking 결과]
  score: 0.0320 | 우리 소설 어떻게 읽을 것인가 4
  score: 0.0217 | 한국 현대 소설의 이해
  score: 0.0120 | 한국 현대소설의 환상과 욕망
  score: 0.0066 | 현대소설의 이야기학
  score: 0.0021 | 소설로 읽는 한국 현대문학 100년 - 대표 장·단편소설 24편 분석
  score: 0.0021 | 가슴에 새긴 너 (2) - 슬픈 남자
  score: 0.0016 | 일대마도 3 - 무적
  score: 0.0015 | 타임리미트 2
  score: 0.0014 | 부활
  score: 0.0012 | 여행 그리고 파리 - Travel and Paris
  [분석] 우리 소설 어떻게 읽을 것인가 4 → 이 책은 한국 현대소설을 배우고 이해하는 데 도움이 되는 자료로, 감상의 폭을 넓히기 위한 다양한 시각을 제...
  [분석] 한국 현

In [ ]:
# app_test

## DeepEval 평가

In [ ]:
# import logging
# logging.getLogger("deepeval").setLevel(logging.WARNING)

# from deepeval import evaluate
# from deepeval.metrics import AnswerRelevancyMetric, FaithfulnessMetric
# from deepeval.test_case import LLMTestCase

# test_cases = [
#     LLMTestCase(
#         input=TEST_SUMMARY,
#         actual_output="\n".join([
#             f"[{b['title']}] {b['reason']}"
#             for b in result["recommendations"]
#         ]),
#         retrieval_context=[
#             f"[{b['title']}] {b['book_intro']}"
#             for b in result["retrieved_books"]
#         ]
#     )
# ]

# evaluate(test_cases, [
#     AnswerRelevancyMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
#     FaithfulnessMetric(threshold=0.7, model="gpt-4o-mini", include_reason=True),
# ])